In [1]:
import sys
import numpy as np

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *

## Get module eigengenes for cell types of interest

In [2]:
ctype_abund_df = pd.read_csv("data/ctype_abundance/GTEx_cortex_counts_TMMF_All_501_outliers_removed_top_Qval_mods_PC1_ctype_abundance_filtered_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules_top_corr_enriched_w_Claude_marker_genes_PC1_ctype_abundance.csv", index_col=0)

In [3]:
data_source = "GTEx_cortex_counts_TMMF_All_501_outliers_removed_top_Qval_mods_PC1_ctype_abundance_filtered_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules_top_corr_enriched_w_Claude_marker_genes_PC1_ctype_abundance"

In [4]:
ctype_abund_df.head()

,CGE Class,All GABAergic,Deep layer glutamatergic,All Neuronal,Oligo,Endo,Peri,OPC,Astro,Micro/PVM,VLMC,Upper layer glutamatergic
Sample,,,,,,,,,,,,
GTEX.12126.0011.R10b.SM.5BC6T,0.144676,0.078921,-0.044717,-0.009510,-0.051565,-0.051868,-0.036232,-0.012470,-0.020109,-0.026597,-0.013267,0.068628
GTEX.12126.0011.R3b.SM.GJ3RE,-0.014015,0.041221,0.045162,-0.008888,-0.027182,-0.066528,-0.067439,0.001876,-0.024565,-0.034164,-0.016592,-0.010536
GTEX.12126.1026.SM.5P9JJ,0.011209,0.051763,0.065797,0.062719,-0.010406,-0.008997,0.032106,-0.003790,-0.030623,-0.010280,0.030002,0.077441
GTEX.12WSE.0011.R3a.SM.GIN9S,0.079146,0.146882,-0.003168,0.041387,-0.036626,-0.028079,0.052985,0.046987,0.006512,-0.031341,0.004336,0.055599
GTEX.15ER7.0011.R3a.SM.6LPIK,0.020049,0.049447,-0.024795,-0.023440,-0.042440,-0.019883,-0.031119,0.007209,0.005926,-0.030310,-0.012150,-0.034865


## Parse GTF (to annotate exons with their genes)

In [5]:
# Parse GTF attribute column
gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.annotation.gtf"
gtf = gtf_parse(gtf_file)
gtf_subset = gtf.loc[gtf['feature'].isin(["gene"])]
attrs = gtf_subset["attribute"].apply(extract_attributes)
attrs_df = attrs.apply(pd.Series)
gtf_parsed = pd.concat([gtf_subset.drop(columns=["attribute"]), attrs_df], axis=1)
gtf_parsed['gene_id'] = gtf_parsed['gene_id'].str.split(".").str[0]

# PSI

In [6]:
psi = pd.read_csv(f"data/GTEx_cortex_exon_PSI.csv", index_col=0)
psi.columns = psi.columns.str.replace("-", ".")

# Make sure order of samples matches 
common = psi.columns.intersection(ctype_abund_df.index)
psi = psi[common]
ctype_abund_df = ctype_abund_df.loc[common]

In [7]:
# Get PSI data ready to merge on gene IDs
psi['gene_id'] = psi.index.str.split("_").str[0]
psi['exon_id'] = psi.index.values

psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
psi_anno = psi_anno.set_index("exon_id").rename_axis(None)
psi_anno = psi_anno.drop(columns=["gene_id"])

In [8]:
# Correlate each cell type with all PSI events across samples

psi_numeric = psi_anno.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

psi_corr_results = {}
for ct in ctype_abund_df.columns:
    psi_corr_results[ct] = psi_numeric.T.corrwith(ctype_abund_df[ct])

psi_corr_df = pd.DataFrame(psi_corr_results)
psi_corr_df.insert(0, 'Gene', psi_anno['gene_name'])

/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/

In [9]:
psi_corr_df.head()

,Gene,CGE Class,All GABAergic,Deep layer glutamatergic,All Neuronal,Oligo,Endo,Peri,OPC,Astro,Micro/PVM,VLMC,Upper layer glutamatergic
ENSG00000292994_other_1,ENSG00000292994,-0.014473,0.032048,0.027809,0.026818,-0.009725,-0.040926,0.000847,-0.015034,-0.070642,-0.089264,0.052085,-0.008983
ENSG00000290385_other_1,ENSG00000290385,-0.015704,0.006412,0.015275,0.072785,0.009770,0.005767,-0.002728,0.006633,0.035221,-0.008543,-0.027174,0.070304
ENSG00000290385_other_2,ENSG00000290385,-0.073088,-0.048841,0.026736,0.024577,0.040311,0.028854,-0.007555,0.002496,0.018016,-0.000776,-0.025606,0.013371
ENSG00000290385_other_3,ENSG00000290385,0.067211,0.085170,0.093188,0.138415,0.048765,0.003060,-0.048024,-0.052080,-0.121479,-0.015591,0.041767,0.086609
ENSG00000290385_other_4,ENSG00000290385,0.093616,0.089313,0.137462,0.271673,0.022167,-0.053059,-0.192378,-0.199484,-0.219776,-0.048427,-0.015869,0.211196


In [10]:
psi_corr_df.to_csv(f"data/corrs/{data_source}_exon_PSI_corr.csv")

# Counts

In [11]:
counts = pd.read_csv(f"data/GTEx_cortex_exon_counts.csv", index_col=0)
counts.columns = counts.columns.str.replace("-", ".")

# Make sure order of samples matches
common = counts.columns.intersection(ctype_abund_df.index)
counts = counts[common]
ctype_abund_df = ctype_abund_df.loc[common]

In [12]:
ctype_abund_df.shape

(501, 12)

In [13]:
# Get PSI and GTF data ready to merge on gene IDs
counts['gene_id'] = counts.index.str.split("_").str[0]
counts['exon_id'] = counts.index.values
counts_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], counts, on="gene_id", how="right")
counts_anno = counts_anno.set_index("exon_id").rename_axis(None)
counts_anno = counts_anno.drop(columns=["gene_id"])

# Correlate each cell type with exon counts

counts_numeric = counts_anno.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

corr_results = {}
for ct in ctype_abund_df.columns:
    corr_results[ct] = counts_numeric.T.corrwith(ctype_abund_df[ct])

corr_df = pd.DataFrame(corr_results)
corr_df.insert(0, 'Gene', counts_anno['gene_name'])

/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/

In [14]:
corr_df.tail()

,Gene,CGE Class,All GABAergic,Deep layer glutamatergic,All Neuronal,Oligo,Endo,Peri,OPC,Astro,Micro/PVM,VLMC,Upper layer glutamatergic
ENSG00000155962_other_2,CLIC2,0.043853,0.060651,0.085269,0.152936,0.139914,0.421942,0.255490,0.168193,0.068312,0.327569,0.394517,0.077746
ENSG00000124333_ProteinCoding_1,VAMP7,0.479061,0.645417,0.275318,0.579811,0.013343,-0.034423,-0.138353,0.157383,0.081824,0.337330,-0.043102,0.379344
ENSG00000124333_ProteinCoding_2,VAMP7,0.498950,0.685315,0.307211,0.640645,0.068934,-0.033802,-0.095495,0.225826,0.109744,0.322322,-0.025900,0.412328
ENSG00000124334_ProteinCoding_1,IL9R,0.022977,0.030601,0.025875,-0.032915,-0.005313,0.046322,0.016725,0.043137,0.032689,0.034725,0.016167,-0.067647
ENSG00000182484_ProteinCoding_1,WASH6P,0.231153,0.284734,0.197912,0.440660,0.089929,-0.020498,-0.061252,-0.032023,-0.052137,0.056050,-0.025107,0.355219


In [15]:
corr_df.to_csv(f"data/corrs/{data_source}_exon_counts_corr.csv")

# Gene expr

In [16]:
bulk_expr = pd.read_csv("GTEx_cortex_counts_TMMF_SampleNetworks/All_02-25-12/GTEx_cortex_counts_TMMF_All_501_outliers_removed.csv")
bulk_expr.columns.values[0] = "Gene"

In [17]:
bulk_expr.head()

,Gene,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y,GTEX.11DXW.0011.R3b.SM.DNZZE,GTEX.11GS4.0011.R3b.SM.GJ3RI,GTEX.11GSO.0011.R3b.SM.57WB2,GTEX.11GSP.0011.R3a.SM.9QEGF,GTEX.11NUK.0011.R3b.SM.GJ3RO,GTEX.11NV4.0011.R3b.SM.GINAJ,...,GTEX.XLM4.0011.R10A.SM.4AT5P,GTEX.Y8DK.0011.R10A.SM.4SOK1,GTEX.YJ89.0011.R10a.SM.4SOK9,GTEX.ZF28.0011.R10a.SM.4WWEH,GTEX.ZUA1.0011.R10a.SM.51MT6,GTEX.ZV68.0011.R10a.SM.51MT7,GTEX.ZVT3.0011.R10b.SM.57WB6,GTEX.ZVZQ.0011.R10b.SM.51MRT,GTEX.ZXG5.0011.R10a.SM.57WDD,GTEX.ZZPT.0011.R10b.SM.GPI8B
0,DDX11L1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.356930,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.903235,0.000000,1.339780,1.004289,0.948680,0.000000,0.000000
1,WASH7P,122.349063,172.427568,130.200106,47.386638,103.159575,36.389517,56.991057,86.216952,177.210769,...,50.181401,83.575447,71.959244,110.194641,53.158136,99.143705,66.283074,79.689127,62.121309,76.635758
2,MIR6859-1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,MIR1302-2HG,1.699293,0.000000,0.000000,0.000000,0.000000,0.933065,1.356930,0.000000,0.000000,...,0.000000,2.785848,0.000000,0.903235,0.000000,2.679560,3.012867,1.897360,1.380474,0.870861
4,FAM138A,0.000000,0.000000,0.000000,2.016453,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2.709704,0.000000,0.000000,1.004289,0.000000,0.000000,0.000000


In [18]:
# Remove duplicate genes by selecting row with highest mean expression

mean_expr = pd.DataFrame({
    'Index': range(len(bulk_expr)),
    'Gene': bulk_expr.iloc[:, 0],
    'Expr': bulk_expr.iloc[:, 1:].mean(axis=1)
})

keep = mean_expr.loc[mean_expr.groupby('Gene')['Expr'].idxmax(), 'Index']
bulk_expr_filtered = bulk_expr.iloc[keep.values]

In [19]:
# Filter 0 genes

mask = bulk_expr_filtered.iloc[:, 1:].mean(axis=1) > 0
bulk_expr_filtered = bulk_expr_filtered[mask]

In [20]:
bulk_expr_filtered = bulk_expr_filtered.set_index("Gene")
common = bulk_expr_filtered.columns.intersection(ctype_abund_df.index)
bulk_expr_filtered = bulk_expr_filtered[common]
ctype_abund_df = ctype_abund_df.loc[common]

In [21]:
ctype_abund_df.shape

(501, 12)

In [22]:
# Correlate each cell type with gene expression

expr_corr_results = {}
for ct in ctype_abund_df.columns:
    expr_corr_results[ct] = bulk_expr_filtered.T.corrwith(ctype_abund_df[ct])
    
expr_corr_df = pd.DataFrame(expr_corr_results)

In [23]:
expr_corr_df.head()

,CGE Class,All GABAergic,Deep layer glutamatergic,All Neuronal,Oligo,Endo,Peri,OPC,Astro,Micro/PVM,VLMC,Upper layer glutamatergic
Gene,,,,,,,,,,,,
5S_rRNA,0.050017,0.113705,0.037459,0.156236,-0.064105,-0.018881,-0.041302,-0.005345,0.019099,0.143568,-0.060981,0.090062
5_8S_rRNA,0.092423,0.127862,0.023519,0.063453,-0.015250,0.000352,-0.037311,0.020922,0.001660,0.074401,-0.026247,0.026789
7SK,0.018007,0.064494,0.055557,0.049035,0.027884,-0.018118,-0.004496,0.026686,0.012173,-0.034212,0.102745,0.008279
A1BG,0.385136,0.535027,0.205803,0.432283,0.200378,0.038310,0.100613,0.464258,0.224893,0.172073,-0.036635,0.251859
A1BG-AS1,0.410480,0.573141,0.466322,0.894040,0.060064,-0.209528,-0.234715,-0.189239,-0.342934,-0.059354,-0.016338,0.688594


In [24]:
expr_corr_df.to_csv(f"data/corrs/{data_source}_gene_expr_corr.csv")